<a href="https://colab.research.google.com/github/krisadas/finrl-multiple/blob/main/FinRLManystock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Mount Google Drive

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive/')

In [5]:
# Install the unstable development version in Jupyter notebook:
!pip install git+https://github.com/AI4Finance-LLC/FinRL-Library.git

  Cloning https://github.com/AI4Finance-LLC/FinRL-Library.git to /tmp/pip-req-build-2i74a60i
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-LLC/FinRL-Library.git /tmp/pip-req-build-2i74a60i
  Resolved https://github.com/AI4Finance-LLC/FinRL-Library.git to commit 083192a2abbdcaa195bd945b35853212cd8b00d8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-uax34g3u/elegantrl_0f1fae577f0b4e75bdf7b5ac7d9bc1b6
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-uax34g3u/elegantrl_0f1fae577f0b4e75bdf7b5ac7d9bc1b6
  Resolved https://github.com/AI4Finance-Foundation/ElegantRL.git to commit c2939fefe0e3ec55601ded49e39fdf9d7d781ea0
  Preparing metadata (setup.py) ... done


In [6]:
import os
import pathlib
import pkg_resources
import pip
installedPackages = {pkg.key for pkg in pkg_resources.working_set}
required = {'yfinance', 'pandas', 'matplotlib', 'stockstats','stable-baselines3','gymnasium','tensorflow'}
missing = required - installedPackages
if missing:
    !pip install yfinance
    !pip install pandas
    !pip install matplotlib
    !pip install stockstats
    !pip install gymnasium
    !pip install stable-baselines3[extra]
    !pip install tensorflow #==1.15.4

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
matplotlib.use('Agg')
import datetime
import os
#from finrl.config import config

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_stock_trading.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent,DRLEnsembleAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

import sys
sys.path.append("../FinRL-Library")

In [15]:
!pip install pyfolio

In [17]:
PATH_TO_MODEL_DIR = 'FinRLManystock/'
print(PATH_TO_MODEL_DIR)

import os
if not os.path.exists(PATH_TO_MODEL_DIR + 'save'):
    os.makedirs(PATH_TO_MODEL_DIR + 'save')
if not os.path.exists(PATH_TO_MODEL_DIR + 'trained'):
    os.makedirs(PATH_TO_MODEL_DIR + 'trained')
if not os.path.exists(PATH_TO_MODEL_DIR + 'tensorboard'):
    os.makedirs(PATH_TO_MODEL_DIR + 'tensorboard')
if not os.path.exists(PATH_TO_MODEL_DIR + 'results'):
    os.makedirs(PATH_TO_MODEL_DIR + 'results')

FinRLManystock/


In [18]:
dow_30_ticker = ['AAPL','MSFT','JPM','V','RTX','PG','GS','NKE','DIS','AXP',
                 'HD','INTC','WMT','IBM','MRK','UNH','KO','CAT','TRV','JNJ', 
                  'CVX','MCD','VZ','CSCO','XOM','BA','MMM','PFE','WBA','DD']
data_df = YahooDownloader(start_date = '2015-01-01',
                          end_date = '2021-01-01',
                          ticker_list = dow_30_ticker).fetch_data()
data_df.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

Shape of DataFrame:  (45330, 8)


,date,open,high,low,close,volume,tic,day
0,2015-01-02,24.347176,27.332500,27.860001,27.847500,212818400,AAPL,4
1,2015-01-02,80.322266,93.019997,93.940002,93.169998,2437500,AXP,4
2,2015-01-02,113.657219,129.949997,131.839996,131.070007,4294200,BA,4
3,2015-01-02,70.367180,91.879997,92.370003,91.769997,3767900,CAT,4
4,2015-01-02,20.326622,27.610001,28.120001,27.860001,22926500,CSCO,4


In [22]:
tech_indicator_list=[]
## you can add more technical indicators
## visit https://github.com/jealous/stockstats for different names
tech_indicator_list=tech_indicator_list+['kdjk','open_2_sma','boll','close','wr_10','dma','trix']
print(tech_indicator_list)

fe = FeatureEngineer(
                    use_technical_indicator=True,
                    tech_indicator_list = tech_indicator_list,
                    use_turbulence=False,
                    user_defined_feature = False)

data_df = fe.preprocess_data(data_df)
data_df.head()

['kdjk', 'open_2_sma', 'boll', 'close', 'wr_10', 'dma', 'trix']
Successfully added technical indicators


,date,open,high,low,close_x,volume,tic,day,kdjk,open_2_sma,boll,close_y,wr_10,dma,trix
0,2015-01-02,24.347176,27.332500,27.860001,27.847500,212818400,AAPL,4,34.123271,24.347176,27.847500,27.847500,-97.630188,0.0,0.0
1511,2015-01-02,80.322266,93.019997,93.940002,93.169998,2437500,AXP,4,61.231863,80.322266,93.169998,93.169998,-16.304411,0.0,0.0
3022,2015-01-02,113.657219,129.949997,131.839996,131.070007,4294200,BA,4,46.913391,113.657219,131.070007,131.070007,-59.259827,0.0,0.0
4533,2015-01-02,70.367180,91.879997,92.370003,91.769997,3767900,CAT,4,74.149617,70.367180,91.769997,91.769997,22.448852,0.0,0.0
6044,2015-01-02,20.326622,27.610001,28.120001,27.860001,22926500,CSCO,4,50.326805,20.326622,27.860001,27.860001,-49.019586,0.0,0.0


In [24]:
df = data_df
# add covariance matrix as states
df=df.sort_values(['date','tic'],ignore_index=True)
df.index = df.date.factorize()[0]

cov_list = []
# look back is one year
lookback=252
for i in range(lookback,len(df.index.unique())):
  data_lookback = df.loc[i-lookback:i,:]
  price_lookback=data_lookback.pivot_table(index = 'date',columns = 'tic', values = 'close_x')
  return_lookback = price_lookback.pct_change().dropna()
  covs = return_lookback.cov().values 
  cov_list.append(covs)
  
df_cov = pd.DataFrame({'date':df.date.unique()[lookback:],'cov_list':cov_list})
df = df.merge(df_cov, on='date')
df = df.sort_values(['date','tic']).reset_index(drop=True)
df.head()        

,date,open,high,low,close_x,volume,tic,day,kdjk,open_2_sma,boll,close_y,wr_10,dma,trix,cov_list
0,2016-01-04,23.860584,26.337500,26.342501,25.652500,270597600,AAPL,0,-15.874316,23.850388,27.756000,25.652500,-181.899292,-1.977800,-0.386834,"[[0.000473231474786466, 0.00011891961334344143..."
1,2016-01-04,59.173836,67.589996,68.180000,68.089996,9248300,AXP,0,43.558311,60.031803,69.676000,68.089996,-103.797631,-2.206600,-0.134199,"[[0.000473231474786466, 0.00011891961334344143..."
2,2016-01-04,126.005104,140.500000,141.699997,141.380005,5719500,BA,0,37.742874,127.839115,144.929002,141.380005,-105.663567,-2.384598,-0.069453,"[[0.000473231474786466, 0.00011891961334344143..."
3,2016-01-04,54.018639,67.989998,68.080002,66.879997,8586900,CAT,0,53.688883,54.006725,67.264500,66.879997,-73.955832,-1.983201,-0.107942,"[[0.000473231474786466, 0.00011891961334344143..."
4,2016-01-04,20.057253,26.410000,26.420000,26.389999,35827100,CSCO,0,47.778483,20.262307,26.991500,26.389999,-102.222272,-0.501400,-0.026222,"[[0.000473231474786466, 0.00011891961334344143..."


In [ ]:
train = data_split(df, start = '2015-01-01', end = '2019-01-01')
trade = data_split(df, start = '2019-01-01', end = '2021-01-01')
train.to_csv(PATH_TO_MODEL_DIR + 'save' + '/train_MULTI.csv',index=False)
trade.to_csv(PATH_TO_MODEL_DIR + 'save' + '/trade_MULTI.csv',index=False)

In [ ]:
stock_dimension = len(train.tic.unique())
state_space = 1 + 2*stock_dimension + len(tech_indicator_list)*stock_dimension
#state_space = 156
print(f"Stock data Dimensions: {stock_dimension}, State Spaces: {state_space}")
env_kwargs = {
    "hmax": 100, 
    "initial_amount": 1000000, 
    #"transaction_cost_pct": 0.001, 
    "buy_cost_pct":0.001,
    "sell_cost_pct":0.001,
    "state_space": state_space, 
    "stock_dim": stock_dimension, 
    "tech_indicator_list": config.TECHNICAL_INDICATORS_LIST, 
    "action_space": stock_dimension, 
    "reward_scaling": 1e-4}
e_train_gym = StockTradingEnv(df = train, **env_kwargs)
env_train, _ = e_train_gym.get_sb_env()
print(type(env_train))

In [ ]:
agent = DRLAgent(env = env_train)
A2C_PARAMS = {"n_steps": 5, "ent_coef": 0.005, "learning_rate": 0.0002}
model_a2c = agent.get_model(model_name="a2c",model_kwargs = A2C_PARAMS)
trained_a2c = agent.train_model(model=model_a2c, 
                                tb_log_name='a2c',
                                total_timesteps=100000)

In [ ]:
trained_a2c.save(PATH_TO_MODEL_DIR + config.TRAINED_MODEL_DIR+'/trained_a2c.model')

In [ ]:
trade.head()

In [ ]:
e_trade_gym = StockTradingEnv(df = trade, **env_kwargs)
df_account_value, df_actions = DRLAgent.DRL_prediction(model=trained_a2c, environment = e_trade_gym)

In [ ]:
print(df_account_value)
df_actions.head()

In [ ]:
from finrl.trade.backtest import backtest_stats, backtest_plot
print("==============Results===========")
now = datetime.datetime.now().strftime('%Y%m%d-%Hh%M')

perf_stats_all = backtest_stats(account_value=df_account_value)
perf_stats_all = pd.DataFrame(perf_stats_all)

In [ ]:
%matplotlib inline
backtest_plot(account_value=df_account_value, baseline_ticker = 'AAPL',
             baseline_start = '2019-01-01', baseline_end = '2021-01-01')